https://www.mlb.com/news/triple-a-abs-challenge-system?msockid=13a1423ec59b64c529f456edc46065f5 -- abs challenge system used full-time in AAA beginning June 25, 2024

# Import Libraries

In [99]:
import polars as pl
import glob
import os

# Load Data

In [161]:
jun = pl.read_csv('data/2024_6_aaa_pbp.csv')
jul = pl.read_csv('data/2024_7_aaa_pbp.csv').with_columns(pl.col('zone').cast(pl.Int64))
aug = pl.read_csv('data/2024_8_aaa_pbp.csv', ignore_errors = True)
# data types are messed up here for some reason
columns_to_cast = ['spin_dir', 'release_spin_rate', 'spin_axis']
aug = aug.with_columns([
    pl.col(col).cast(pl.Float64) for col in columns_to_cast
])
sep = pl.read_csv('data/2024_9_aaa_pbp.csv').with_columns(pl.col('zone').cast(pl.Int64))
pbp = pl.concat([jun, jul, aug, sep]).unique()
pbp = pbp.with_columns(
    pl.col("play_start_datetime").str.to_datetime('%Y-%m-%d %H:%M:%S%.3f')
)
pbp = pbp.filter(pl.col('play_start_datetime') >= pl.datetime(2024, 6, 25))
pbp = pbp.with_columns(pl.arange(1, pbp.height + 1).alias('id')).with_row_index()
# filter to games with challenge system

In [102]:
pbp.head()

index,play_start_datetime,play_end_datetime,pitch_type,pitch_name,game_date,release_speed,release_pos_x,release_pos_y,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,…,fielder_7,fielder_8,fielder_9,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,game_month,game_day,game_year,league_id,league_name,league_level_id,league_level_name,away_team_org_id,away_team_org_name,home_team_org_id,home_team_org_name,id
u32,datetime[ms],str,str,str,str,f64,f64,f64,f64,str,i64,i64,str,str,f64,str,str,str,i64,str,str,str,str,str,str,str,f64,str,i64,i64,f64,f64,f64,f64,f64,f64,…,i64,i64,i64,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,f64,str,str,i64,i64,i64,i64,str,i64,str,i64,str,i64,str,i64
0,2024-06-25 22:36:52.474,"""2024-06-25 22:36:56.921""","""SI""","""Sinker""","""2024-06-25""",94.2,-2.42476,50.003853,5.322163,"""Will Warren""",805367,701542,null,"""Chase Meidroth reaches on a th…",235.0,null,null,null,7,"""Chase Meidroth reaches on a th…","""R""","""R""","""R""","""SWB""","""WOR""","""C""",null,null,0,1,-11.605821,2.447198,-0.459962,1.542261,null,null,…,666211,663604,670351,null,null,null,null,null,null,null,0,1,0,0,0,0,0,0,0,0,null,null,235.0,null,null,6,25,2024,117,"""International League""",11,"""Triple-A""",111,"""Boston Red Sox""",147,"""New York Yankees""",1
1,2024-06-25 22:37:04.300,"""2024-06-25 22:37:08.331""","""FF""","""Four-Seam Fastball""","""2024-06-25""",93.9,-2.29617,50.00311,5.354985,"""Will Warren""",805367,701542,null,"""Chase Meidroth reaches on a th…",234.0,null,null,null,3,"""Chase Meidroth reaches on a th…","""R""","""R""","""R""","""SWB""","""WOR""","""C""",null,null,0,2,-8.399939,8.131848,0.826216,2.839424,null,null,…,666211,663604,670351,null,null,null,null,null,null,null,0,2,0,0,0,0,0,0,0,0,null,null,234.0,null,null,6,25,2024,117,"""International League""",11,"""Triple-A""",111,"""Boston Red Sox""",147,"""New York Yankees""",2
2,2024-06-25 22:37:15.811,"""2024-06-25 22:37:19.586""","""ST""","""Sweeper""","""2024-06-25""",83.4,-2.591875,50.000042,5.444827,"""Will Warren""",805367,701542,null,"""Chase Meidroth reaches on a th…",65.0,null,null,null,14,"""Chase Meidroth reaches on a th…","""R""","""R""","""R""","""SWB""","""WOR""","""B""",null,null,1,2,6.651188,0.411169,1.206732,0.997029,null,null,…,666211,663604,670351,null,null,null,null,null,null,null,0,3,0,0,0,0,0,0,0,0,null,null,65.0,null,null,6,25,2024,117,"""International League""",11,"""Triple-A""",111,"""Boston Red Sox""",147,"""New York Yankees""",3
3,2024-06-25 22:37:29.854,"""2024-06-25 22:37:32.955""","""FF""","""Four-Seam Fastball""","""2024-06-25""",94.5,-2.35674,50.000621,5.435499,"""Will Warren""",805367,701542,null,"""Chase Meidroth reaches on a th…",229.0,null,null,null,11,"""Chase Meidroth reaches on a th…","""R""","""R""","""R""","""SWB""","""WOR""","""F""",null,null,1,2,-7.619783,8.237165,-0.067982,3.437228,null,null,…,666211,663604,670351,null,null,null,null,null,null,null,0,4,0,0,0,0,0,0,0,0,null,null,229.0,null,null,6,25,2024,117,"""International League""",11,"""Triple-A""",111,"""Boston Red Sox""",147,"""New York Yankees""",4
4,2024-06-25 22:37:50.548,"""2024-06-25 22:37:54.300""","""SI""","""Sinker""","""2024-06-25""",94.2,-2.106085,50.002089,5.333808,"""Will Warren""",805367,701542,null,"""Chase Meidroth reaches on a th…",235.0,null,null,null,6,"""Chase Meidroth reaches on a th…","""R""","""R""","""R""","""SWB""","""WOR""","""F""",null,null,1,2,-10.102835,-0.34487,0.326928,2.052791,null,null,…,6662

# Filter to Pitches with Challenges

In [72]:
ab_with_challenge = pbp.filter(pl.col('description').str.contains('challenge'))
# last pitch of at bat: group by home team, away team, inning, top / bottom, at bat, and select last pitch 
group_cols = ["home_team", "away_team", "game_date", "inning", "inning_top_bot", "at_bat_number"]
pitch_with_challenge = (
    ab_with_challenge.sort("pitch_number")
    .group_by(group_cols)
    .agg(pl.col("*").last())
)

In [73]:
pitch_with_challenge.head()

home_team,away_team,game_date,inning,inning_top_bot,at_bat_number,index,play_start_datetime,play_end_datetime,pitch_type,pitch_name,release_speed,release_pos_x,release_pos_y,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,type,hit_location,bb_type,balls,strikes,pfx_x,pfx_z,plate_x,…,fielder_6,fielder_7,fielder_8,fielder_9,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,pitch_number,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,game_month,game_day,game_year,league_id,league_name,league_level_id,league_level_name,away_team_org_id,away_team_org_name,home_team_org_id,home_team_org_name,id
str,str,str,i64,str,i64,u32,datetime[ms],str,str,str,f64,f64,f64,f64,str,i64,i64,str,str,f64,str,str,str,i64,str,str,str,str,str,f64,str,i64,i64,f64,f64,f64,…,i64,i64,i64,i64,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,f64,str,str,i64,i64,i64,i64,str,i64,str,i64,str,i64,str,i64
"""NAS""","""MEM""","""2024-09-11""",1,"""Top""",2,598479,2024-09-11 23:42:33.895,"""2024-09-11 23:43:05.512""","""CH""","""Changeup""",85.8,-1.181399,50.002839,5.562042,"""Carlos Rodriguez""",663845,692230,null,"""Redbirds challenged (pitch res…",247.0,null,null,null,9,"""Redbirds challenged (pitch res…","""R""","""L""","""R""","""B""",null,null,4,2,-10.566372,2.071211,0.656084,…,668965,677578,676896,676551,null,null,null,null,null,null,null,6,0,0,0,0,0,0,0,0,null,null,247.0,null,null,9,11,2024,117,"""International League""",11,"""Triple-A""",138,"""St. Louis Cardinals""",158,"""Milwaukee Brewers""",598480
"""JAX""","""NAS""","""2024-06-26""",9,"""Top""",87,9917,2024-06-26 19:04:36.614,"""2024-06-26 19:04:53.617""","""FF""","""Four-Seam Fastball""",93.7,-0.965496,50.000983,5.827638,"""Luarbert Arias""",686555,678215,null,"""Sounds challenged (pitch resul…",215.0,null,null,null,1,"""Sounds challenged (pitch resul…","""R""","""L""","""R""","""C""",null,null,2,3,-6.319624,9.158079,-0.571593,…,656484,664753,683748,665052,null,null,null,null,null,null,null,5,10,10,10,10,10,10,10,10,null,null,215.0,null,null,6,26,2024,117,"""International League""",11,"""Triple-A""",158,"""Milwaukee Brewers""",146,"""Miami Marlins""",9918
"""SYR""","""DUR""","""2024-08-21""",7,"""Bot""",63,428865,2024-08-22 00:43:50.360,"""2024-08-22 00:43:54.061""","""SL""","""Slider""",84.2,-1.502487,50.000619,5.343357,"""Alfredo Zarraga""",682239,700287,null,"""Bulls challenged (pitch result…",75.0,null,null,null,3,"""Bulls challenged (pitch result…","""R""","""R""","""R""","""C""",null,null,1,3,0.648612,1.077256,0.728747,…,678545,681715,671976,678010,null,null,null,null,null,null,null,4,4,6,4,6,6,4,4,6,null,null,75.0,null,null,8,21,2024,117,"""International League""",11,"""Triple-A""",139,"""Tampa Bay Rays""",121,"""New York Mets""",428866
"""LHV""","""WOR""","""2024-07-21""",6,"""Bot""",55,187786,2024-07-21 19:26:53.313,"""2024-07-21 19:27:14.761""","""FF""","""Four-Seam Fastball""",92.2,-1.748674,50.004262,5.494357,"""Brian Van Belle""",665561,687003,null,"""IronPigs challenged (pitch res…",217.0,null,null,null,8,"""IronPigs challenged (pitch res…","""R""","""L""","""R""","""C""",null,null,2,3,-3.105158,8.305104,0.072242,…,686765,683090,676628,666915,null,null,null,null,null,null,null,6,6,1,6,1,1,6,6,1,null,null,217.0,null,null,7,21,2024,117,"""International League""",11,"""Triple-A""",111,"""Boston Red Sox""",143,"""Philadelphia Phillies""",187787
"""ABQ""","""SAC""","""2024-08-06""",8,"""Top""",78,313848,2024-08-07 03:11:29.557,"""2024-08-07 03:11:56.567""","""FC""","""Cutter""",90.5,-2.642968,50.004291,5.961536,"""Geoff Hartlieb""",670092,664129,null,"""River Cats challenged (pitch r…",204.0,null,null,null,11,"""River Cats challenged (p

In [74]:
# upheld vs overturned
pitch_with_challenge = pitch_with_challenge.with_columns(
    pl.when(pl.col('description').str.contains(r"(?i)overturned"))
    .then(1)
    .otherwise(0)
    .alias('challenge_successful')
)
# in September 2024, 51 successful challenges and 195 unsuccessful challenges
pitch_with_challenge.select(pl.col('challenge_successful').value_counts())

challenge_successful
struct[2]
"{0,740}"
"{1,188}"


In [75]:
# c: called strike
# s: strikes out swinging?
pitch_with_challenge.select(pl.col('type').value_counts())

type
struct[2]
"{""S"",2}"
"{""B"",305}"
"{""C"",621}"


In [76]:
pitch_with_challenge['type']

type
str
"""B"""
"""C"""
"""C"""
"""C"""
"""B"""
…
"""C"""
"""C"""
"""C"""


In [77]:
# Challenge by offense vs pitching
# 1 = challenge by pitching team, 0 = challenge by batter
# the pitching team should challenge more!
pitch_with_challenge = pitch_with_challenge.with_columns(
    # called ball, overturned to strike
    pl.when((pl.col("type").is_in(['S', 'C'])) & (pl.col("challenge_successful") == 1))
    .then(1)
    # called strike, upheld
    .when((pl.col("type").is_in(['S', 'C'])) & (pl.col("challenge_successful") == 0))
    .then(0)
    # called strike, overturned to ball
    .when((pl.col("type") == "B") & (pl.col("challenge_successful") == 1))
    .then(0)
    # called ball, upheld
    .when((pl.col("type") == "B") & (pl.col("challenge_successful") == 0))
    .then(1)
    .alias("challenge_by")
)
pitch_with_challenge.select(pl.col('challenge_by').value_counts())

challenge_by
struct[2]
"{0,661}"
"{1,267}"


# Understanding the 'type' column

In [162]:
with pl.Config(tbl_rows = 100):
    print(pbp.group_by(pl.col('type')).len().sort('len', descending = True))

shape: (17, 2)
┌──────┬────────┐
│ type ┆ len    │
│ ---  ┆ ---    │
│ str  ┆ u32    │
╞══════╪════════╡
│ B    ┆ 118950 │
│ F    ┆ 59104  │
│ C    ┆ 51821  │
│ S    ┆ 37706  │
│ X    ┆ 35162  │
│ D    ┆ 13231  │
│ E    ┆ 8319   │
│ *B   ┆ 6597   │
│ T    ┆ 3351   │
│ W    ┆ 1622   │
│ H    ┆ 1064   │
│ L    ┆ 681    │
│ M    ┆ 116    │
│ O    ┆ 19     │
│ P    ┆ 18     │
│ R    ┆ 1      │
│ Z    ┆ 1      │
└──────┴────────┘


In [163]:
# Create a game id column to find unique at-bats
game_id_lookup = (
    pbp
        .select(['game_date', 'home_team_org_id', 'away_team_org_id'])
        .unique()
        .with_row_index()
        .rename({'index': 'game_id'})
)
pbp = pbp.join(game_id_lookup, on = ['game_date', 'home_team_org_id', 'away_team_org_id'])

In [164]:
# Create a game id column to find unique at-bats
at_bat_id_lookup = (
    pbp
        .select(['game_id', 'inning', 'inning_top_bot', 'batter', 'pitcher', 'outs_when_up'])
        .unique()
        .with_row_index()
        .rename({'index': 'at_bat_id'})
)
pbp = pbp.join(at_bat_id_lookup, on = ['game_id', 'inning', 'inning_top_bot', 'batter', 'pitcher', 'outs_when_up'])

## "B"
Ball -- when it's the first pitch of an at-bat, almost always 1-0 with the exception of a few 2-0 and 1-1 counts, likely due to pitch clock violations

In [165]:
# B type
(
    pbp
        .filter(
            (pl.col('type') == 'B') &
            (pl.col('pitch_number') == 1)
        )
        .group_by(['balls', 'strikes'])
        .len()
)

balls,strikes,len
i64,i64,u32
1,1,12
2,0,56
1,0,33390


## "F"
Foul ball -- almost always an 0-1 count when it's the first pitch of an at-bat (occasional pitch-clock violations)

In [166]:
(
    pbp
        .filter(
            (pl.col('type') == 'F') &
            (pl.col('pitch_number') == 1)
        )
        .group_by(['balls', 'strikes'])
        .len()
)

balls,strikes,len
i64,i64,u32
0,1,10259
0,2,7
1,1,17


## C
Called strike -- always a strike, and when it's the last pitch of the at-bat it's always a called strikeout

In [173]:
# find previous balls and strikes
# partition by game_id, at_bat_number and order by pitch_number
pbp = (
    pbp
        .with_columns([
            pl.col('balls')
            .shift(1)
            .over(
                partition_by = ['game_id', 'at_bat_id'],
                order_by = 'pitch_number'
            )
            .fill_null(strategy = 'zero')
            .alias('lag_balls'),
            pl.col('strikes')
            .shift(1)
            .over(
                partition_by = ['game_id', 'at_bat_id'],
                order_by = 'pitch_number'
            )
            .fill_null(strategy = 'zero')
            .alias('lag_strikes')
        ]
        )
)

In [177]:
(
    pbp
        .filter(
            (pl.col('game_id') == 476) &
            (pl.col('at_bat_id') == 83724)
        )
        .select([
        'pitch_number',
        'balls',
        'strikes',
        'lag_balls',
        'lag_strikes'
        ])
        .sort('pitch_number')
)

pitch_number,balls,strikes,lag_balls,lag_strikes,type
i64,i64,i64,i64,i64,str
1,0,1,0,0,"""C"""
2,0,2,0,1,"""S"""
3,0,2,0,2,"""F"""
4,0,2,0,2,"""F"""
5,0,2,0,2,"""F"""
6,1,2,0,2,"""B"""
7,1,2,1,2,"""F"""
8,2,2,1,2,"""B"""
9,3,2,2,2,"""B"""


In [179]:
# always a strike 
(
    pbp
        .filter(pl.col('type') == 'C')
        .with_columns(
            pl.when(pl.col('strikes') > pl.col('lag_strikes'))
            .then(1)
            .otherwise(0)
            .alias('strike')
        )
        .group_by('strike')
        .len()
)

strike,len
i32,u32
1,51821


In [202]:
# whenever the type is C and the pitch is the third strike, the batter is called out on strikes
(
    pbp
        .filter(
            (pl.col('type') == 'C') &
            (pl.col('strikes') == 3)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

['Carlos Narvaez called out on strikes.',
 'Andrew Navigato called out on strikes.',
 'Bligh Madris called out on strikes.',
 'IronPigs challenged (pitch result), call on the field was upheld: Ryan McKenna called out on strikes.',
 'Ruben Cardenas called out on strikes.',
 'Redbirds challenged (pitch result), call on the field was upheld: Thomas Saggese called out on strikes.',
 'Gabriel Cancel called out on strikes.',
 'Vinny Capra called out on strikes.',
 'Sam Huff called out on strikes.',
 'Chris Roller called out on strikes.']

## S
swinging strike

In [201]:
(
    pbp
        .filter(
            (pl.col('type') == 'S') &
            (pl.col('strikes') == 3)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

['Andre Lipcius strikes out swinging.',
 'Luis De Los Santos strikes out swinging.',
 'Connor Kaiser strikes out swinging.',
 'Trevor Boone strikes out swinging.',
 'Bob Seymour strikes out swinging.',
 'Daniel Johnson strikes out swinging.',
 'Kyle Garlick strikes out swinging.',
 'Kevin Alcántara strikes out swinging.',
 'Justice Bigbie strikes out swinging.',
 'Will Robertson strikes out swinging.']

## X
ball in play (data dictionary)

## D, #, 

Seems like this is a ball put in play where the runner reaches base -- not sure how it differs from X

In [193]:
# did this for each column -- always the last pitch of an at bat with the ball in play, but the result seems random
a = (
    pbp
        .filter(pl.col('type') == "T")
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes',
            'pitch_type',
            'des'
        ])
)

## *B
Ball

In [192]:
(
    pbp
        .filter(pl.col('type') == "*B")
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes'
            ])
)

balls,strikes,lag_balls,lag_strikes
i64,i64,i64,i64
4,2,3,2
2,2,1,2
2,1,1,1
3,0,2,0
1,0,0,0
…,…,…,…
1,1,0,1
2,2,1,2
1,1,0,1


## T

Foul tip -- always a strike and description shows it's a foul tip

In [199]:
(
    pbp
        .filter(
            (pl.col('type') == "T") &
            (pl.col('strikes') == 3)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

['Will Robertson strikes out on a foul tip.',
 'Kody Hoese strikes out on a foul tip.',
 'Deyvison De Los Santos strikes out on a foul tip.',
 'Drew Avans strikes out on a foul tip.',
 'Drew Waters strikes out on a foul tip.',
 'Anthony Bemboom strikes out on a foul tip.',
 'Edmundo Sosa strikes out on a foul tip.',
 'Drake Baldwin strikes out on a foul tip.',
 'Cooper Bowman strikes out on a foul tip.',
 'Yanquiel Fernandez strikes out on a foul tip.']

## W
Seems like it's also a strike swining -- maybe 'whiff'?

In [ ]:
# always a strike
(
    pbp
        .filter(pl.col('type') == "W")
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes'
            ])
)

balls,strikes,lag_balls,lag_strikes
i64,i64,i64,i64
0,3,0,2
1,3,1,2
1,2,1,1
1,1,1,0
2,3,2,2
…,…,…,…
3,3,3,2
1,1,1,0
2,2,2,1


In [204]:
(
    pbp
        .filter(
            (pl.col('type') == "W") &
            (pl.col('strikes') == 3)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

['Tim Elko strikes out swinging.',
 'Bryce Teodosio strikes out swinging.',
 'Bryce Windham strikes out swinging.',
 'Nick Pratto strikes out swinging.',
 'Owen Caissie strikes out swinging.',
 'Gavin Collins strikes out swinging.',
 'Shane Matheny strikes out swinging, catcher Brett Sullivan to first baseman Tirso Ornelas.',
 'Kevin Smith strikes out swinging.',
 'Jake Lamb strikes out swinging.',
 'Carlos Pérez strikes out swinging.']

## H

Hit by pitch

In [ ]:
# always a ball
(
    pbp
        .filter(pl.col('type') == "H")
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes'
            ])
)

balls,strikes,lag_balls,lag_strikes
i64,i64,i64,i64
1,1,0,1
1,2,0,2
1,1,0,1
2,2,1,2
3,2,2,2
…,…,…,…
3,2,2,2
4,2,3,2
1,2,0,2


In [206]:
(
    pbp
        .filter(
            (pl.col('type') == "H") &
            (pl.col('balls') == 4)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

['Tyler Locklear hit by pitch.',
 'Anthony Prato hit by pitch.    Chris Williams to 2nd.',
 'Cam Devanney hit by pitch.',
 'Luke Ritter hit by pitch.',
 'Andrew Navigato hit by pitch.',
 'Dylan Crews hit by pitch.',
 'Seth Beer hit by pitch.    Joshua Palacios to 3rd.    Matt Gorski to 2nd.',
 'Payton Eeles hit by pitch.',
 'Logan Porter hit by pitch.',
 'Connor Kaiser hit by pitch.']

## M
Definitely a strike, but unclear whether the batter swung or not

In [ ]:
# always a strike
(
    pbp
        .filter(pl.col('type') == "M")
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes'
            ])
)

balls,strikes,lag_balls,lag_strikes
i64,i64,i64,i64
1,2,1,1
0,1,0,0
0,1,0,0
1,1,1,0
0,1,0,0
…,…,…,…
0,1,0,0
3,2,3,1
0,2,0,1


In [219]:
# never a strikeout
(
    pbp
        .filter(
            (pl.col('type') == "M") &
            (pl.col('strikes') == 3)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

[]

In [ ]:
# this pitch was about 9 inches outside but a strike -- could this be a called strike or was it definitely swinging?
(
    pbp
        .filter(
            (pl.col('type') == 'M')
        )
        .select(pl.col('plate_x').min())
)

plate_x
f64
-1.584805


## O

Similar to M -- definitely a strike, but unknown if the batter swung or not

In [ ]:
# always a strike
(
    pbp
        .filter(pl.col('type') == "O")
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes'
            ])
)

balls,strikes,lag_balls,lag_strikes
i64,i64,i64,i64
0,1,0,0
0,2,0,1
0,1,0,0
0,1,0,0
1,1,1,0
…,…,…,…
0,1,0,0
0,1,0,0
0,1,0,0


In [223]:
# never a strikeout
(
    pbp
        .filter(
            (pl.col('type') == "O") &
            (pl.col('strikes') == 3)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

[]

## P
ball, but never a walk

In [226]:
# always a strike
(
    pbp
        .filter(pl.col('type') == "P")
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes'
            ])
)

balls,strikes,lag_balls,lag_strikes
i64,i64,i64,i64
1,0,0,0
2,1,1,1
1,1,0,1
2,0,1,0
1,0,0,0
…,…,…,…
1,1,0,1
2,0,1,0
1,0,0,0


In [ ]:
# never a walk
(
    pbp
        .filter(
            (pl.col('type') == "P") &
            (pl.col('balls') == 4)
        )
        .select('des')
        .to_series()
        .to_list()
        [:10]
)

[]

## Z and R
one-off data errors -- unclear

In [ ]:
# always a strike
(
    pbp
        .filter(pl.col('type').is_in(['Z', 'R']))
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes',
            'type',
            'des'
            ])
)

balls,strikes,lag_balls,lag_strikes,type,des
i64,i64,i64,i64,str,str
1,2,1,1,"""R""","""Phil Clarke walks. Damiano P…"
2,2,2,2,"""Z""","""Jordan Diaz singles on a groun…"


# Identifying Potential Success of Challenges

# Classify Pitch as in- or out-of-zone
x is centered at 0 and measured in feet. Home plate is 17 inches wide, and the strike zone's horizontal limits include home plate plus the diameter of the baseball (2.94 inches) = 19.94 inches = 1.66 feet. Thus, the limits of the strike zone are +- 0.83.

In [ ]:
# columns: sz_top, sz_bot, plate_x, plate_z
# calculate on pbp data to identify missed challenge opportunities
import numpy as np
pbp = pbp.with_columns(
    pl.when(
        (np.abs(pl.col('plate_x')) < 0.83) &
        (pl.col('plate_z') >= pl.col('sz_bot')) &
        (pl.col('plate_z') <= pl.col('sz_top'))
    )
    .then(1)
    .otherwise(0)
    .alias('in_strike_zone')
)

In [ ]:
pbp.select(pl.col('in_strike_zone').value_counts()).unnest('in_strike_zone')

in_strike_zone,count
i32,u32
0,395476
1,299686


## Identify Missed Calls

In [ ]:
# assuming B is a ball and S is a taken strike, but not sure because there's so many values
with pl.Config(tbl_rows = 17):
    print(pbp.group_by('type').len().sort('len', descending = True))

shape: (17, 2)
┌──────┬────────┐
│ type ┆ len    │
│ ---  ┆ ---    │
│ str  ┆ u32    │
╞══════╪════════╡
│ B    ┆ 244892 │
│ F    ┆ 121834 │
│ C    ┆ 106548 │
│ S    ┆ 77646  │
│ X    ┆ 72308  │
│ D    ┆ 27222  │
│ E    ┆ 17098  │
│ *B   ┆ 13486  │
│ T    ┆ 6920   │
│ W    ┆ 3318   │
│ H    ┆ 2176   │
│ L    ┆ 1400   │
│ M    ┆ 234    │
│ O    ┆ 40     │
│ P    ┆ 36     │
│ R    ┆ 2      │
│ Z    ┆ 2      │
└──────┴────────┘


In [ ]:
# type: when b and 1, or s and 0
# because of issues with the data, I can only identify called strikeouts for now (C) as we can't know if the batter swung at the pitch or not
pbp = (
    pbp.with_columns(
        pl.when(
        # called a ball, but in the strike zone
        (
            (pl.col('type') == 'C') &
            (pl.col('in_strike_zone') == 0)
) |
        # called a strike, but not in the strike zone
        (
            (pl.col('type') == 'B') &
            (pl.col('in_strike_zone') == 1)
        ))
        .then(1)
        .otherwise(0)
        .alias('missed_call')
    )
)

In [ ]:
challenged_pitches = pitch_with_challenge['id'].to_list()

In [ ]:
# why are there so many 'missed calls' that aren't successful challenges?
pbp.join(pitch_with_challenge.select(pl.col('id'), pl.col('challenge_successful')), on = 'id').group_by(['missed_call', 'challenge_successful']).len()

missed_call,challenge_successful,len
i32,i32,u32
0,1,162
1,1,26
1,0,100
0,0,640


In [ ]:
challenged_merge = pbp.join(pitch_with_challenge.select(pl.col('id'), pl.col('challenge_successful')), on = 'id')

In [ ]:
pitch_locs = challenged_merge.select(['plate_x', 'plate_z', 'sz_bot', 'sz_top', 'in_strike_zone', 'type', 'missed_call', 'challenge_successful'])

In [ ]:
challenge_discrepancies = pitch_locs.filter(pl.col('missed_call') != pl.col('challenge_successful'))
# seems like there's pitches that seem to be baove the strike zone but aren't being labeled as such
# could look for roster info and fill in from there?
# experiment with adding/multiplying values
pitch_height_discrepancies = challenge_discrepancies.filter(
    (pl.col('plate_x') < 0.84) &
    (pl.col('plate_x') > -0.84)
)

# Questions to pursue if we can't figure out the data issues here:
1) Which teams use the most challenges?
2) Which types of pitches are challenged most often?
3) Do teams use their challenges on high run-value pitches?
4) Create player cards for catchers / batters on where they challenge pitches